# Étape 3 — Nettoyage et dédoublonnage exact

Objectif : visualiser chaque transformation, mesurer les suppressions et comparer le résultat au traitement réutilisable de `00_fonctions_base.ipynb`.

Toutes les opérations sont effectuées en mémoire : la base et ses textes originaux restent inchangés. Ce notebook n'ajoute aucun filtrage d'actualité.

## 1. Chargement des données

In [ ]:
from pathlib import Path
import sys

# Fonctionne depuis le notebook, le sous-projet ou la racine du projet.
cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "bluesky-news-mining", cwd / "textmining" / "bluesky-news-mining"]
ROOT = next((p.resolve() for p in candidates if (p / "notebooks" / "00_fonctions_base.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ouvrir le notebook depuis le projet Bluesky ou son dossier notebooks.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATABASE_PATH = ROOT / "posts.db"
print("Sous-projet :", ROOT.name)
print("Base SQLite :", (Path(ROOT.name) / DATABASE_PATH.relative_to(ROOT)).as_posix())

In [ ]:
import pandas as pd
from IPython.display import display

import json

# Charger les fonctions communes, sans lire ni modifier la base.
base_path = ROOT / "notebooks" / "00_fonctions_base.ipynb"
base_notebook = json.loads(base_path.read_text(encoding="utf-8"))
for definition_cell in base_notebook["cells"]:
    if definition_cell["cell_type"] == "code" and "database-definitions" in definition_cell["metadata"].get("tags", []):
        exec(compile("".join(definition_cell["source"]), str(base_path), "exec"), globals())

pd.set_option("display.max_colwidth", 120)
posts_raw = load_posts(DATABASE_PATH, limit=None)
print("Publications chargées :", len(posts_raw))

## 2. Vérification de la langue déclarée

On conserve les posts dont `langs` contient exactement `fr`. Il s'agit du même contrôle que lors de la collecte ; un post `en,fr` est accepté.

In [ ]:
is_french = posts_raw["langs"].fillna("").str.split(",").map(lambda langs: "fr" in langs).astype(bool)
posts_fr = posts_raw.loc[is_french].copy()
print("Publications déclarées en français :", len(posts_fr))

## 3. Nettoyage par expressions régulières

La fonction `clean_text` supprime les URL commençant par `http://`, `https://` ou `www.`, normalise les espaces et retire les espaces aux extrémités. La casse, la ponctuation, les accents et les emojis sont conservés. Il n'y a ni lemmatisation ni suppression des mots vides.

La colonne `text` conserve l'original ; `clean_text` reçoit la version nettoyée.

In [ ]:
example = "  Grève demain !\nhttps://exemple.fr/article\t Rendez-vous à Lyon.  "
print("Avant :", repr(example))
print("Après :", repr(clean_text(example)))

posts_cleaned = posts_fr.copy()
posts_cleaned["clean_text"] = posts_cleaned["text"].map(clean_text)
display(posts_cleaned[["text", "clean_text"]].head(10))

## 4. Retrait des textes trop courts

Après suppression des URL, un post peut être vide ou ne plus contenir que quelques caractères. Le seuil actuel est de 3 caractères minimum ; il ne s'agit pas d'un nombre de mots.

In [ ]:
long_enough = posts_cleaned["clean_text"].str.len() >= MIN_TEXT_LENGTH
posts_long = posts_cleaned.loc[long_enough].copy()
print("Longueur minimale :", MIN_TEXT_LENGTH, "caractères")
print("Publications retirées pour longueur insuffisante :", len(posts_cleaned) - len(posts_long))

## 5. Dédoublonnage exact des textes nettoyés

Deux lignes sont considérées comme doublons si leur `clean_text` est strictement identique. La comparaison est sensible à la casse et à la ponctuation. Deux textes ayant des URL différentes peuvent devenir identiques après nettoyage.

On conserve la première occurrence chargée, donc la plus récente selon la date utilisée au chargement (avec l'URI comme second critère de tri). Les doublons restent dans SQLite : seule la table en mémoire est réduite.

In [ ]:
duplicate_mask = posts_long["clean_text"].duplicated(keep="first")
print("Occurrences dupliquées retirées :", int(duplicate_mask.sum()))
display(posts_long.loc[posts_long["clean_text"].duplicated(keep=False), ["uri", "text", "clean_text"]].head(10))

posts_unique = posts_long.drop_duplicates(subset="clean_text", keep="first").reset_index(drop=True)
print("Publications nettoyées et uniques :", len(posts_unique))

## 6. Bilan des étapes

In [ ]:
counts = [
    ("Chargement", len(posts_raw)),
    ("Langue déclarée fr", len(posts_fr)),
    ("Nettoyage des URL et espaces", len(posts_cleaned)),
    ("Longueur minimale", len(posts_long)),
    ("Dédoublonnage exact", len(posts_unique)),
]
summary = pd.DataFrame(counts, columns=["Étape", "Publications"])
summary["Retirées à cette étape"] = (summary["Publications"].shift(1) - summary["Publications"]).fillna(0).astype(int)
display(summary)
display(posts_unique[["created_at", "did", "text", "clean_text"]].head(10))

## 7. Vérifier le résultat

Le traitement détaillé doit donner exactement le même résultat que `preprocess_posts`. On vérifie aussi l'absence de doublons et le seuil de longueur.

In [ ]:
expected = preprocess_posts(posts_raw)
pd.testing.assert_frame_equal(posts_unique, expected)
assert posts_unique["clean_text"].is_unique
assert (posts_unique["clean_text"].str.len() >= MIN_TEXT_LENGTH).all()
assert len(load_posts(DATABASE_PATH, limit=None)) == len(posts_raw), "Le nombre de lignes de la base a changé pendant le notebook."
print("Résultat identique à preprocess_posts : OK.")
print("Absence de doublons et longueur minimale : OK.")
print("Publications brutes :", len(posts_raw), "| conservées :", len(posts_unique))